# Sesi 11 — RAG bagian 2: Retrieval dan Optimasi

Notebook sebelumnya memasukkan tujuh dokumen ke dalam index. Notebook ini
mengambilnya kembali, lalu memperbaiki cara pengambilannya.

**Retrieval adalah kebalikan dari ingestion.** Di notebook 1 kita mengubah teks
menjadi vector lalu menyimpannya. Di sini kita mengubah pertanyaan menjadi vector,
mencari vector terdekat, lalu menerjemahkannya kembali menjadi teks.

Bagian kedua notebook ini adalah optimasi, yaitu hybrid search dan re-ranker. Dua
teknik yang dipakai ketika similarity search saja belum cukup presisi.

In [ ]:
import json
import os
import pathlib

import faiss
import numpy as np
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv()

MODEL_EMBED = "gemini-embedding-001"
DIMENSI = 768


def akar_sesi():
    # Folder yang memuat kb_lindungiku, entah folder ini atau folder induknya.
    for p in (pathlib.Path("."), pathlib.Path("..")):
        if (p / "kb_lindungiku").is_dir():
            return p.resolve()
    raise SystemExit(
        "Folder kb_lindungiku tidak ditemukan. Jalankan notebook ini dari folder "
        "Session 11 atau dari salah satu subfoldernya."
    )


AKAR = akar_sesi()
BERKAS_INDEX = AKAR / "index_lindungiku.faiss"
BERKAS_POTONGAN = AKAR / "potongan.json"

if not BERKAS_INDEX.exists():
    raise SystemExit(
        f"{BERKAS_INDEX} belum ada. Jalankan rag_ingestion terlebih dahulu, "
        "atau minta berkas siap pakai kepada mentor."
    )

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

index = faiss.read_index(str(BERKAS_INDEX))
potongan = json.loads(BERKAS_POTONGAN.read_text(encoding="utf-8"))

print("vector di index :", index.ntotal)
print("potongan dimuat :", len(potongan))
assert index.ntotal == len(potongan), "index dan potongan.json tidak sinkron, jalankan ulang rag_ingestion"

## Langkah 1 — Bertanya, mengambil, membaca

Dua fungsi yang sama seperti di notebook 1, dengan satu perbedaan penting.

`task_type` sekarang `RETRIEVAL_QUERY`, bukan `RETRIEVAL_DOCUMENT`. Model embedding
harus sama pada kedua sisi, tetapi tipe tugasnya berbeda. Dokumen disimpan sebagai
sesuatu yang akan dicari, pertanyaan dikirim sebagai sesuatu yang mencari.

`index.search` mengembalikan dua larik, yaitu skor dan nomor baris. Karena vector
kita sudah dinormalkan, skor inner product ini sama persis dengan cosine
similarity. Rentangnya secara teori dari -1 sampai 1, dan pada teks nyata hampir
selalu berada di antara 0 dan 1.

In [ ]:
def normalisasi(v):
    panjang = np.linalg.norm(v, axis=1, keepdims=True)
    panjang[panjang == 0] = 1e-12
    return v / panjang


def embed(daftar_teks, task_type):
    resp = client.models.embed_content(
        model=MODEL_EMBED,
        contents=daftar_teks,
        config=types.EmbedContentConfig(task_type=task_type, output_dimensionality=DIMENSI),
    )
    return np.array([e.values for e in resp.embeddings], dtype="float32")


def cari(pertanyaan, k=5):
    # TODO 1: ubah pertanyaan menjadi vector, normalkan, lalu cari k tetangga terdekat.
    # Ingat: task_type untuk pertanyaan adalah "RETRIEVAL_QUERY", bukan RETRIEVAL_DOCUMENT.
    # index.search(q, k) mengembalikan (skor, baris).
    # Kembalikan daftar dict berisi skor, sumber, dan teks.
    pass


PERTANYAAN = "dokumen apa saja yang wajib diunggah untuk klaim rawat inap"

for n, h in enumerate(cari(PERTANYAAN), 1):
    print(f"#{n}  {h['skor']:.3f}  {h['sumber'][:2]}  {h['teks'].strip()[:66]}")

### Pentingnya membaca hasil retrieval

Pertanyaannya tentang klaim **kesehatan**. Perhatikan kolom sumber di atas.

Apakah ada potongan dari sumber yang tidak relevan, misal dari `02_kebijakan_klaim_kendaraan.md` 
yang ikut naik ?. Dokumen tersebut  memakai kosakata yang hampir sama, yaitu *klaim*, *dokumen
wajib*, *hari kerja*, *verifikasi*, *pencairan*. Bagi model embedding kedua
dokumen terlihat sangat mirip, karena memang mirip. Yang berbeda hanya satu hal,
yaitu pertanyaannya tentang yang mana.

**Inilah masalah yang akan kita perbaiki di bagian 11.4.** Dokumen yang mirip (similiar) belum tentu
relevan/ Pencarian menemukan yang mirip, bukan yang menjawab.

Jalankan sel di bawah untuk melihat sebaran sumbernya dengan jelas.

In [ ]:
hasil = cari(PERTANYAAN, k=5)
salah = [h for h in hasil if not h["sumber"].startswith("01")]

print(f"pertanyaan : {PERTANYAAN}")
print(f"dokumen yang seharusnya menjawab : 01_kebijakan_klaim_kesehatan.md")
print()
for n, h in enumerate(hasil, 1):
    tanda = "  <-- dokumen lain" if not h["sumber"].startswith("01") else ""
    print(f"#{n}  {h['skor']:.3f}  {h['sumber']}{tanda}")
print()
print(f"{len(salah)} dari 5 hasil teratas berasal dari dokumen yang salah.")
print("Itulah noise yang akan dibersihkan oleh re-ranker.")

## Langkah 2 — Hybrid search

Ada satu jenis pertanyaan yang selalu gagal dijawab oleh semantic search, yaitu
pertanyaan yang mengandung kode.

Kode polis seperti `LK-PRM-24` tidak memiliki makna yang bisa di-embed. Bagi model
embedding, string itu hanyalah rangkaian karakter yang tidak mirip dengan apa pun.
Tetapi bagi pencarian kata kunci, kode itu justru yang paling mudah ditemukan,
karena cocok secara persis.

Keyword search dan semantic search gagal di tempat yang berbeda. Itulah alasan
memakai keduanya sekaligus.

**BM25** adalah algoritma pencarian kata kunci klasik. Ia menghitung seberapa
sering sebuah kata muncul di satu dokumen dibandingkan seluruh koleksi. Kata yang
langka dan muncul berkali-kali di satu dokumen mendapat bobot tinggi. Kode polis
adalah kasus sempurna untuk BM25.

In [ ]:
# perlu install beberapa library di sini
# pip instal sentence-transformers langchain-community rank_bm25

In [ ]:
from langchain_core.documents import Document
from langchain_core.retrievers import BaseRetriever
from langchain_community.retrievers import BM25Retriever

dokumen_lc = [
    Document(page_content=p["teks"], metadata={"sumber": p["sumber"], "baris": i})
    for i, p in enumerate(potongan)
]

# TODO 2: buat retriever kata kunci dari dokumen_lc memakai BM25Retriever.from_documents,
# lalu atur bm25.k = 5 supaya ia mengembalikan lima hasil.
bm25 = None


class PencariFaiss(BaseRetriever):
    k: int = 5

    def _get_relevant_documents(self, query, *, run_manager=None):
        # TODO 3: embed query dengan RETRIEVAL_QUERY, normalkan, cari self.k terdekat,
        # lalu kembalikan dokumen_lc pada nomor baris hasil pencarian.
        pass


semantik = PencariFaiss(k=5)
print("bm25 siap, semantik siap")

### Kita cek dengan mencoba mencari suatu kode produk unik

In [ ]:
UJI = "syarat produk LK-PRM-24"

def ringkas(judul, daftar):
    print(judul)
    for n, d in enumerate(daftar, 1):
        ada = "KODE ADA" if "LK-PRM-24" in d.page_content else "        "
        print(f"   #{n} [{ada}] {d.metadata['sumber'][:2]}  {d.page_content.strip()[:58]}")
    print()

ringkas("SEMANTIC SEARCH saja", semantik.invoke(UJI))
ringkas("KEYWORD SEARCH saja", bm25.invoke(UJI))

### Menggabungkan keduanya

Cara menggabungkan dua daftar peringkat yang skornya tidak sebanding disebut
**Reciprocal Rank Fusion**. 

Setiap dokumen mendapat nilai berdasarkan **peringkatnya**, bukan skornya:

```
nilai(d) = jumlah dari  bobot_i / (60 + peringkat_i)
```

Peringkat satu memberi sumbangan besar, peringkat sepuluh memberi sumbangan kecil.
Angka 60 adalah konstanta peredam yang lazim dipakai. Karena yang dipakai adalah
peringkat, kita tidak perlu memaksa skor BM25 dan skor cosine berada pada skala
yang sama, dan memang keduanya tidak sebanding.

In [ ]:
try:
    from langchain.retrievers import EnsembleRetriever
    asal = "langchain.retrievers"
except ImportError:
    from langchain_classic.retrievers import EnsembleRetriever
    asal = "langchain_classic.retrievers"

print("EnsembleRetriever diimpor dari", asal)

# TODO 4: gabungkan bm25 dan semantik memakai EnsembleRetriever.
# Beri bobot 0.4 untuk kata kunci dan 0.6 untuk makna, lalu panggil .invoke(UJI).
hybrid = None
hasil_hybrid = None

ringkas("HYBRID, keyword dan semantic digabung", hasil_hybrid[:5])

## Langkah 3 — Re-ranker

**Mengapa similarity search bisa meleset.** Vector pertanyaan dan vector potongan
dibuat **secara terpisah**, masing-masing tanpa mengetahui yang lain. Model
embedding memampatkan seluruh makna sebuah potongan menjadi 768 angka sebelum ada
pertanyaan apa pun. Sebagian konteks pasti hilang di situ.

**Apa yang dikerjakan cross-encoder.** Ia membaca pertanyaan dan potongan
**bersamaan dalam satu masukan**, lalu mengeluarkan satu angka, yaitu seberapa
cocok pasangan ini. Tidak ada pemampatan. Karena itu jauh lebih akurat.

**Harga yang harus dibayar.** Satu pemanggilan model per pasangan. Untuk 44
potongan berarti 44 pemanggilan, bayangkan jika kita punya 1 juta dokumen. Karena itu
polanya selalu dua tahap: ambil lima sampai dua puluh kandidat dengan pencarian
yang murah, lalu urutkan ulang kandidat itu saja dengan model yang mahal.

In [ ]:
from sentence_transformers import CrossEncoder

MODEL_RERANK = "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1"
reranker = CrossEncoder(MODEL_RERANK)


def rerank(pertanyaan, kandidat, k=3):
    # TODO 5: susun pasangan (pertanyaan, isi potongan) untuk setiap kandidat,
    # nilai semuanya dengan reranker.predict, lalu urutkan dari nilai tertinggi.
    # Kembalikan k teratas sebagai daftar tuple (nilai, dokumen).
    pass


kandidat = [
    Document(page_content=h["teks"], metadata={"sumber": h["sumber"]})
    for h in cari(PERTANYAAN, k=5)
]

print("SEBELUM re-ranking")
for n, d in enumerate(kandidat, 1):
    print(f"   #{n}  {d.metadata['sumber'][:2]}  {d.page_content.strip()[:60]}")

print()
print("SESUDAH re-ranking")
for n, (nilai, d) in enumerate(rerank(PERTANYAAN, kandidat, k=5), 1):
    print(f"   #{n}  {nilai:+.2f}  {d.metadata['sumber'][:2]}  {d.page_content.strip()[:52]}")

### Quick check

Model re-ranker yang paling sering direkomendasikan di internet adalah
`cross-encoder/ms-marco-MiniLM-L6-v2`. Model itu bagus, banyak dipakai, dan
angkanya di berbagai benchmark meyakinkan.

Model tersebut dilatih **hanya dengan bahasa Inggris**. MS MARCO adalah dataset
peringkat teks berbahasa Inggris.

Jalankan sel di bawah dan bandingkan sendiri. Perhatikan sebaran nilainya, bukan
nilai satuannya.

In [ ]:
model_en = CrossEncoder("cross-encoder/ms-marco-MiniLM-L6-v2")

pasangan = [(PERTANYAAN, d.page_content) for d in kandidat]
nilai_en = model_en.predict(pasangan)
nilai_id = reranker.predict(pasangan)

print(f"{'sumber':<10} {'inggris':>10} {'multilingual':>14}")
print("-" * 36)
for d, a, b in zip(kandidat, nilai_en, nilai_id):
    print(f"{d.metadata['sumber'][:2]:<10} {a:>10.3f} {b:>14.3f}")

print()
print(f"sebaran nilai model inggris      : {float(np.ptp(nilai_en)):.3f}")
print(f"sebaran nilai model multilingual : {float(np.ptp(nilai_id)):.3f}")
print()
print("Sebaran yang sempit berarti model tidak bisa membedakan kandidat.")
print("Urutan yang dihasilkannya mendekati acak.")